
# 🏆 PRONÓSTICO ESTADÍSTICO AVANZADO - MUNDIAL 2026
## Uzbekistán vs Colombia | Grupo K | 17/06/2026 20:00

**Autor:** Anselmo Salguero | Maestría en Estadística Aplicada

**Metodología:**
- Dataset histórico internacional (49,398 partidos desde 1872)
- Modelo de Poisson para goles esperados
- Simulación Monte Carlo (10,000 iteraciones)
- Análisis de plantillas y valor de mercado
- Datos en tiempo real desde API Mundial 2026

**Fuentes:**
- API REST: `worldcup26.ir`
- Dataset histórico: GitHub `martj42/international_results`
- Plantillas: ESPN / Transfermarkt


In [1]:

# =============================================================================
# CELDA 1: IMPORTACIÓN DE LIBRERÍAS
# =============================================================================
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import poisson
import warnings
warnings.filterwarnings('ignore')

# Configuración visual - Paleta Pastel Profesional
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

COLORES = {
    'celeste': '#A8D8EA', 'naranja': '#FFD3B6', 'amarillo': '#FFF9C4',
    'verde': '#C8E6C9', 'rosa': '#F8BBD0', 'lavanda': '#E1BEE7',
    'gris': '#F5F5F5', 'colombia': '#FCD116', 'colombia_azul': '#003893',
    'uzbekistan': '#1EB53A', 'uzbekistan_azul': '#0099B5'
}

print("✅ Librerías importadas correctamente")
print("📦 pandas:", pd.__version__)
print("📦 numpy:", np.__version__)
print("📦 matplotlib:", plt.matplotlib.__version__)


✅ Librerías importadas correctamente
📦 pandas: 2.3.3
📦 numpy: 2.3.5
📦 matplotlib: 3.10.6



## 📁 SECCIÓN 1: DATASET HISTÓRICO INTERNACIONAL
Descargamos el dataset de GitHub `martj42/international_results` que contiene **49,398 partidos** desde 1872 hasta 2024.


In [2]:

# =============================================================================
# CELDA 2: DESCARGA DATASET HISTÓRICO
# =============================================================================
URL_HISTORICO = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"

print("📡 Descargando dataset histórico...")
df_hist = pd.read_csv(URL_HISTORICO)

print(f"✅ Dataset cargado: {len(df_hist):,} partidos")
print(f"📅 Rango de fechas: {df_hist['date'].min()} a {df_hist['date'].max()}")
print(f"
📊 Primeras filas:")
print(df_hist.head())
print(f"
📋 Columnas: {list(df_hist.columns)}")
print(f"
📈 Partidos por década:")
df_hist['year'] = pd.to_datetime(df_hist['date']).dt.year
decadas = (df_hist['year'] // 10 * 10).value_counts().sort_index()
print(decadas.tail(10))


SyntaxError: unterminated f-string literal (detected at line 11) (3913923142.py, line 11)


## 🔍 SECCIÓN 2: FILTRADO POR EQUIPOS
Extraemos todos los partidos históricos de Colombia y Uzbekistán para calcular estadísticas de rendimiento.


In [ ]:

# =============================================================================
# CELDA 3: FILTRAR PARTIDOS POR EQUIPO
# =============================================================================
# Partidos de Colombia
colombia_home = df_hist[df_hist['home_team'] == 'Colombia'].copy()
colombia_away = df_hist[df_hist['away_team'] == 'Colombia'].copy()
colombia_all = pd.concat([colombia_home, colombia_away]).sort_values('date')

# Partidos de Uzbekistán
uzbekistan_home = df_hist[df_hist['home_team'] == 'Uzbekistan'].copy()
uzbekistan_away = df_hist[df_hist['away_team'] == 'Uzbekistan'].copy()
uzbekistan_all = pd.concat([uzbekistan_home, uzbekistan_away]).sort_values('date')

# Enfrentamientos directos
directos = df_hist[
    ((df_hist['home_team'] == 'Colombia') & (df_hist['away_team'] == 'Uzbekistan')) |
    ((df_hist['home_team'] == 'Uzbekistan') & (df_hist['away_team'] == 'Colombia'))
].copy()

print("="*70)
print("📊 COLOMBIA - HISTORIAL INTERNACIONAL")
print("="*70)
print(f"   Total partidos: {len(colombia_all)}")
print(f"   Como local: {len(colombia_home)}")
print(f"   Como visitante: {len(colombia_away)}")
print(f"   Primero: {colombia_all['date'].min()}")
print(f"   Último: {colombia_all['date'].max()}")

print(f"
{'='*70}")
print("📊 UZBEKISTÁN - HISTORIAL INTERNACIONAL")
print(f"{'='*70}")
print(f"   Total partidos: {len(uzbekistan_all)}")
print(f"   Como local: {len(uzbekistan_home)}")
print(f"   Como visitante: {len(uzbekistan_away)}")
print(f"   Primero: {uzbekistan_all['date'].min()}")
print(f"   Último: {uzbekistan_all['date'].max()}")

print(f"
{'='*70}")
print("⚔️ ENFRENTAMIENTOS DIRECTOS")
print(f"{'='*70}")
print(f"   Total partidos: {len(directos)}")
if len(directos) > 0:
    print(directos[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']].to_string(index=False))
else:
    print("   ❌ No hay enfrentamientos directos históricos registrados")



## 📈 SECCIÓN 3: ESTADÍSTICAS HISTÓRICAS POR EQUIPO
Calculamos goles promedio, victorias, empates y derrotas para cada equipo en su rol de local y visitante.


In [ ]:

# =============================================================================
# CELDA 4: ESTADÍSTICAS HISTÓRICAS DETALLADAS
# =============================================================================

def calcular_stats_equipo(df_home, df_away, nombre_equipo):
    """Calcula estadísticas completas de un equipo."""
    # Como local
    goles_favor_local = df_home['home_score'].sum()
    goles_contra_local = df_home['away_score'].sum()
    victorias_local = len(df_home[df_home['home_score'] > df_home['away_score']])
    empates_local = len(df_home[df_home['home_score'] == df_home['away_score']])
    derrotas_local = len(df_home[df_home['home_score'] < df_home['away_score']])
    pj_local = len(df_home)

    # Como visitante
    goles_favor_away = df_away['away_score'].sum()
    goles_contra_away = df_away['home_score'].sum()
    victorias_away = len(df_away[df_away['away_score'] > df_away['home_score']])
    empates_away = len(df_away[df_away['away_score'] == df_away['home_score']])
    derrotas_away = len(df_away[df_away['away_score'] < df_away['home_score']])
    pj_away = len(df_away)

    # Totales
    pj_total = pj_local + pj_away
    gf_total = goles_favor_local + goles_favor_away
    gc_total = goles_contra_local + goles_contra_away

    stats = {
        'equipo': nombre_equipo,
        'pj_total': pj_total, 'pj_local': pj_local, 'pj_visitante': pj_away,
        'gf_total': gf_total, 'gc_total': gc_total,
        'gf_local': goles_favor_local, 'gc_local': goles_contra_local,
        'gf_visitante': goles_favor_away, 'gc_visitante': goles_contra_away,
        'prom_gf_total': gf_total / pj_total if pj_total > 0 else 0,
        'prom_gc_total': gc_total / pj_total if pj_total > 0 else 0,
        'prom_gf_local': goles_favor_local / pj_local if pj_local > 0 else 0,
        'prom_gc_local': goles_contra_local / pj_local if pj_local > 0 else 0,
        'prom_gf_visitante': goles_favor_away / pj_away if pj_away > 0 else 0,
        'prom_gc_visitante': goles_contra_away / pj_away if pj_away > 0 else 0,
        'victorias_local': victorias_local, 'empates_local': empates_local, 'derrotas_local': derrotas_local,
        'victorias_visitante': victorias_away, 'empates_visitante': empates_away, 'derrotas_visitante': derrotas_away,
        'pct_v_local': victorias_local / pj_local * 100 if pj_local > 0 else 0,
        'pct_e_local': empates_local / pj_local * 100 if pj_local > 0 else 0,
        'pct_d_local': derrotas_local / pj_local * 100 if pj_local > 0 else 0,
        'pct_v_visitante': victorias_away / pj_away * 100 if pj_away > 0 else 0,
        'pct_e_visitante': empates_away / pj_away * 100 if pj_away > 0 else 0,
        'pct_d_visitante': derrotas_away / pj_away * 100 if pj_away > 0 else 0,
    }
    return stats

# Calcular para ambos equipos
stats_col = calcular_stats_equipo(colombia_home, colombia_away, 'Colombia')
stats_uzb = calcular_stats_equipo(uzbekistan_home, uzbekistan_away, 'Uzbekistan')

# Mostrar comparativa
print("="*70)
print("📊 COMPARATIVA ESTADÍSTICA HISTÓRICA")
print("="*70)
print(f"
{'Indicador':<35} {'Colombia':>15} {'Uzbekistan':>15}")
print("-"*70)
print(f"{'Partidos jugados (total)':<35} {stats_col['pj_total']:>15,} {stats_uzb['pj_total']:>15,}")
print(f"{'Partidos como local':<35} {stats_col['pj_local']:>15,} {stats_uzb['pj_local']:>15,}")
print(f"{'Partidos como visitante':<35} {stats_col['pj_visitante']:>15,} {stats_uzb['pj_visitante']:>15,}")
print(f"{'Goles a favor (total)':<35} {stats_col['gf_total']:>15,} {stats_uzb['gf_total']:>15,}")
print(f"{'Goles en contra (total)':<35} {stats_col['gc_total']:>15,} {stats_uzb['gc_total']:>15,}")
print(f"{'Prom. goles a favor (local)':<35} {stats_col['prom_gf_local']:>15.2f} {stats_uzb['prom_gf_local']:>15.2f}")
print(f"{'Prom. goles en contra (local)':<35} {stats_col['prom_gc_local']:>15.2f} {stats_uzb['prom_gc_local']:>15.2f}")
print(f"{'Prom. goles a favor (visitante)':<35} {stats_col['prom_gf_visitante']:>15.2f} {stats_uzb['prom_gf_visitante']:>15.2f}")
print(f"{'Prom. goles en contra (visitante)':<35} {stats_col['prom_gc_visitante']:>15.2f} {stats_uzb['prom_gc_visitante']:>15.2f}")
print(f"{'% Victoria como local':<35} {stats_col['pct_v_local']:>14.1f}% {stats_uzb['pct_v_local']:>14.1f}%")
print(f"{'% Victoria como visitante':<35} {stats_col['pct_v_visitante']:>14.1f}% {stats_uzb['pct_v_visitante']:>14.1f}%")
print(f"{'% Empate como local':<35} {stats_col['pct_e_local']:>14.1f}% {stats_uzb['pct_e_local']:>14.1f}%")
print(f"{'% Empate como visitante':<35} {stats_col['pct_e_visitante']:>14.1f}% {stats_uzb['pct_e_visitante']:>14.1f}%")



## 🌐 SECCIÓN 4: DATOS EN TIEMPO REAL - MUNDIAL 2026
Conectamos con la API REST oficial del torneo para obtener el fixture actualizado.


In [ ]:

# =============================================================================
# CELDA 5: API MUNDIAL 2026 - DATOS EN TIEMPO REAL
# =============================================================================
BASE_URL = "http://worldcup26.ir"

print("📡 Conectando con API Mundial 2026...")
games_resp = requests.get(f"{BASE_URL}/get/games", timeout=30)
games_data = games_resp.json()['games']
df_games = pd.DataFrame(games_data)

teams_resp = requests.get(f"{BASE_URL}/get/teams", timeout=30)
df_teams = pd.DataFrame(teams_resp.json()['teams'])

# Buscar partido específico
partido = df_games[
    ((df_games['home_team_name_en'].str.contains('Uzbekistan', case=False, na=False)) & 
     (df_games['away_team_name_en'].str.contains('Colombia', case=False, na=False)))
].iloc[0]

team_local = partido['home_team_name_en']
team_visitante = partido['away_team_name_en']

print(f"
✅ PARTIDO ENCONTRADO:")
print(f"   {team_local} vs {team_visitante}")
print(f"   Grupo: {partido['group']}")
print(f"   Fecha: {partido['local_date']}")
print(f"   Estado: {partido['time_elapsed']}")
print(f"   Estadio ID: {partido['stadium_id']}")

# Estadísticas del torneo actual
finished = df_games[df_games['finished'] == 'TRUE'].copy()
finished['home_score_num'] = pd.to_numeric(finished['home_score'], errors='coerce')
finished['away_score_num'] = pd.to_numeric(finished['away_score'], errors='coerce')

stats_torneo = {
    'total_partidos': len(finished),
    'promedio_goles_total': (finished['home_score_num'].sum() + finished['away_score_num'].sum()) / len(finished) if len(finished) > 0 else 3.0,
    'promedio_goles_local': finished['home_score_num'].mean() if len(finished) > 0 else 2.0,
    'promedio_goles_visitante': finished['away_score_num'].mean() if len(finished) > 0 else 1.0,
}

print(f"
📊 Estadísticas del torneo actual ({stats_torneo['total_partidos']} partidos finalizados):")
print(f"   Promedio goles/partido: {stats_torneo['promedio_goles_total']:.2f}")
print(f"   Local: {stats_torneo['promedio_goles_local']:.2f}")
print(f"   Visitante: {stats_torneo['promedio_goles_visitante']:.2f}")



## 🔬 SECCIÓN 5: MODELO PREDICTIVO - POISSON AJUSTADO
Combinamos datos históricos con datos del torneo actual para calcular λ (goles esperados) de cada equipo.

**Fórmula:**
```
λ = (promedio_base) × (factor_localía) × (factor_historico_ofensivo) × (factor_historico_defensivo_oponente)
```


In [ ]:

# =============================================================================
# CELDA 6: MODELO POISSON CON DATOS HISTÓRICOS + TORNEO ACTUAL
# =============================================================================

# FACTORES DE FUERZA basados en datos históricos
if stats_col['pj_visitante'] > 20:
    factor_ofensivo_col = stats_col['prom_gf_visitante'] / max(stats_torneo['promedio_goles_visitante'], 0.5)
    factor_defensivo_col = stats_col['prom_gc_visitante'] / max(stats_torneo['promedio_goles_local'], 0.5)
else:
    factor_ofensivo_col = 1.0
    factor_defensivo_col = 1.0

if stats_uzb['pj_local'] > 20:
    factor_ofensivo_uzb = stats_uzb['prom_gf_local'] / max(stats_torneo['promedio_goles_local'], 0.5)
    factor_defensivo_uzb = stats_uzb['prom_gc_local'] / max(stats_torneo['promedio_goles_visitante'], 0.5)
else:
    factor_ofensivo_uzb = 1.0
    factor_defensivo_uzb = 1.0

# Factor de localía conservador
factor_localia = 1.20

# Calcular lambdas
lambda_local = stats_torneo['promedio_goles_local'] * factor_localia * factor_ofensivo_uzb * (1 / max(factor_defensivo_col, 0.5))
lambda_visitante = stats_torneo['promedio_goles_visitante'] / factor_localia * factor_ofensivo_col * (1 / max(factor_defensivo_uzb, 0.5))

# Ajustar valores extremos
lambda_local = max(min(lambda_local, 4.0), 0.5)
lambda_visitante = max(min(lambda_visitante, 3.0), 0.5)

print("="*70)
print("🔬 PARÁMETROS DEL MODELO POISSON AJUSTADO")
print("="*70)
print(f"
📊 Factores de ajuste:")
print(f"   Factor localía: {factor_localia}")
print(f"   Factor ofensivo {team_local} (histórico): {factor_ofensivo_uzb:.3f}")
print(f"   Factor defensivo {team_local} (histórico): {factor_defensivo_uzb:.3f}")
print(f"   Factor ofensivo {team_visitante} (histórico): {factor_ofensivo_col:.3f}")
print(f"   Factor defensivo {team_visitante} (histórico): {factor_defensivo_col:.3f}")
print(f"
⚽ Parámetros λ (goles esperados):")
print(f"   λ({team_local}) = {lambda_local:.3f}")
print(f"   λ({team_visitante}) = {lambda_visitante:.3f}")
print(f"
📈 Total goles esperados: {lambda_local + lambda_visitante:.2f}")



## 📊 SECCIÓN 6: MATRIZ DE PROBABILIDADES Y RESULTADO 1X2
Calculamos la probabilidad de cada marcador posible y las probabilidades agregadas de victoria local, empate y victoria visitante.


In [ ]:

# =============================================================================
# CELDA 7: MATRIZ DE PROBABILIDADES Y 1X2
# =============================================================================
max_goles = 7
matriz = np.zeros((max_goles + 1, max_goles + 1))

for i in range(max_goles + 1):
    for j in range(max_goles + 1):
        matriz[i, j] = poisson.pmf(i, lambda_local) * poisson.pmf(j, lambda_visitante)

# Probabilidades 1X2
prob_local = sum(matriz[i, j] for i in range(max_goles+1) for j in range(max_goles+1) if i > j)
prob_empate = sum(matriz[i, j] for i in range(max_goles+1) for j in range(max_goles+1) if i == j)
prob_visitante = sum(matriz[i, j] for i in range(max_goles+1) for j in range(max_goles+1) if i < j)

prob_1x2 = {'1': prob_local, 'X': prob_empate, '2': prob_visitante}

print("="*70)
print("📈 PROBABILIDADES 1X2 (Modelo Poisson)")
print("="*70)
print(f"
   1 ({team_local}): {prob_1x2['1']*100:.2f}%")
print(f"   X (Empate):     {prob_1x2['X']*100:.2f}%")
print(f"   2 ({team_visitante}): {prob_1x2['2']*100:.2f}%")

# Tabla de resultados más probables
print(f"
{'='*70}")
print("📋 TOP 15 RESULTADOS MÁS PROBABLES")
print(f"{'='*70}")
resultados_tabla = []
for i in range(max_goles + 1):
    for j in range(max_goles + 1):
        prob = matriz[i, j] * 100
        if prob > 0.1:
            resultados_tabla.append({
                'Marcador': f"{i}-{j}",
                'Resultado': '1' if i > j else ('X' if i == j else '2'),
                'Probabilidad': prob
            })

df_resultados = pd.DataFrame(resultados_tabla)
df_resultados = df_resultados.sort_values('Probabilidad', ascending=False)
print(df_resultados.head(15).to_string(index=False))



## 🎲 SECCIÓN 7: SIMULACIÓN MONTE CARLO
Simulamos 10,000 partidos virtuales para estimar la distribución de resultados con intervalos de confianza.


In [ ]:

# =============================================================================
# CELDA 8: SIMULACIÓN MONTE CARLO (10,000 ITERACIONES)
# =============================================================================
np.random.seed(42)
n_sim = 10000

goles_l = np.random.poisson(lambda_local, n_sim)
goles_v = np.random.poisson(lambda_visitante, n_sim)

mc = pd.DataFrame({
    'goles_local': goles_l,
    'goles_visitante': goles_v
})
mc['resultado'] = mc.apply(
    lambda r: '1' if r['goles_local'] > r['goles_visitante'] else 
              ('X' if r['goles_local'] == r['goles_visitante'] else '2'), axis=1)
mc['total_goles'] = mc['goles_local'] + mc['goles_visitante']
mc['diferencia'] = mc['goles_local'] - mc['goles_visitante']

probs_mc = mc['resultado'].value_counts(normalize=True)

print("="*70)
print(f"🎲 MONTE CARLO - {n_sim:,} SIMULACIONES")
print("="*70)
for res, prob in probs_mc.items():
    nombre = team_local if res == '1' else (team_visitante if res == '2' else 'Empate')
    print(f"   {res} ({nombre}): {prob*100:.2f}%")

# Marcador más probable
top_marcador = mc.groupby(['goles_local', 'goles_visitante']).size().sort_values(ascending=False).head(1)
marcador_prob = (top_marcador.values[0] / n_sim) * 100
gl, gv = top_marcador.index[0]

print(f"
🏆 Marcador más probable: {gl}-{gv} ({marcador_prob:.1f}%)")
print(f"
📊 Estadísticas adicionales:")
print(f"   Media goles {team_local}: {mc['goles_local'].mean():.2f}")
print(f"   Media goles {team_visitante}: {mc['goles_visitante'].mean():.2f}")
print(f"   Media total goles: {mc['total_goles'].mean():.2f}")
print(f"   Over 2.5: {((mc['total_goles'] > 2.5).mean()*100):.1f}%")
print(f"   Under 2.5: {((mc['total_goles'] <= 2.5).mean()*100):.1f}%")
print(f"   Ambos anotan: {(((mc['goles_local'] > 0) & (mc['goles_visitante'] > 0)).mean()*100):.1f}%")
print(f"   Solo local anota: {(((mc['goles_local'] > 0) & (mc['goles_visitante'] == 0)).mean()*100):.1f}%")
print(f"   Solo visitante anota: {(((mc['goles_local'] == 0) & (mc['goles_visitante'] > 0)).mean()*100):.1f}%")



## 📊 SECCIÓN 8: VISUALIZACIONES PROFESIONALES

### Gráfico 1: Matriz de Probabilidades (Heatmap)


In [ ]:

# =============================================================================
# CELDA 9: GRÁFICO 1 - MATRIZ DE PROBABILIDADES
# =============================================================================
fig, ax = plt.subplots(figsize=(12, 10))

annot = np.empty_like(matriz, dtype=object)
for i in range(matriz.shape[0]):
    for j in range(matriz.shape[1]):
        annot[i, j] = f"{matriz[i, j]*100:.1f}%"

sns.heatmap(matriz, annot=annot, fmt='', cmap='YlOrRd', 
            cbar_kws={'label': 'Probabilidad'},
            linewidths=1, linecolor='white',
            ax=ax, vmin=0, vmax=matriz.max())

ax.set_xlabel(f'Goles {team_visitante} (Visitante)', fontsize=13, fontweight='bold')
ax.set_ylabel(f'Goles {team_local} (Local)', fontsize=13, fontweight='bold')
ax.set_title(
    f'MATRIZ DE PROBABILIDADES - RESULTADOS POSIBLES
'
    f'{team_local} vs {team_visitante} | Mundial 2026 | Grupo K
'
    f'λ({team_local}) = {lambda_local:.2f} | λ({team_visitante}) = {lambda_visitante:.2f}',
    fontsize=14, fontweight='bold', pad=20
)

# Resaltar diagonal (empates)
for i in range(min(matriz.shape)):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor='#003893', lw=3))

plt.tight_layout()
plt.show()



### Gráfico 2: Distribución de Poisson por Equipo


In [ ]:

# =============================================================================
# CELDA 10: GRÁFICO 2 - DISTRIBUCIÓN DE GOLES (POISSON)
# =============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

goles_range = range(0, 8)

# Local
probs_local = [poisson.pmf(k, lambda_local) * 100 for k in goles_range]
bars1 = ax1.bar(goles_range, probs_local, color=COLORES['verde'], 
                edgecolor='white', linewidth=2, width=0.7)
ax1.set_xlabel('Goles', fontsize=12, fontweight='bold')
ax1.set_ylabel('Probabilidad (%)', fontsize=12, fontweight='bold')
ax1.set_title(f'{team_local} (Local)
λ = {lambda_local:.2f} goles esperados', 
              fontsize=13, fontweight='bold')
ax1.set_xticks(list(goles_range))
ax1.set_ylim(0, max(probs_local) * 1.25)
ax1.grid(axis='y', alpha=0.3)

for bar, prob in zip(bars1, probs_local):
    height = bar.get_height()
    ax1.annotate(f'{prob:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# Visitante
probs_visitante = [poisson.pmf(k, lambda_visitante) * 100 for k in goles_range]
bars2 = ax2.bar(goles_range, probs_visitante, color=COLORES['amarillo'], 
                edgecolor='white', linewidth=2, width=0.7)
ax2.set_xlabel('Goles', fontsize=12, fontweight='bold')
ax2.set_ylabel('Probabilidad (%)', fontsize=12, fontweight='bold')
ax2.set_title(f'{team_visitante} (Visitante)
λ = {lambda_visitante:.2f} goles esperados', 
              fontsize=13, fontweight='bold')
ax2.set_xticks(list(goles_range))
ax2.set_ylim(0, max(probs_visitante) * 1.25)
ax2.grid(axis='y', alpha=0.3)

for bar, prob in zip(bars2, probs_visitante):
    height = bar.get_height()
    ax2.annotate(f'{prob:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('DISTRIBUCIÓN DE POISSON - GOLES ESPERADOS', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()



### Gráfico 3: Dashboard Simulación Monte Carlo


In [ ]:

# =============================================================================
# CELDA 11: GRÁFICO 3 - MONTE CARLO (4 PANELES)
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Pie 1X2
ax1 = axes[0, 0]
probs = mc['resultado'].value_counts(normalize=True) * 100
colores_pie = [COLORES['verde'], COLORES['amarillo'], COLORES['naranja']]
labels_pie = [f'1 ({team_local}): {probs.get("1",0):.1f}%', 
              f'X (Empate): {probs.get("X",0):.1f}%', 
              f'2 ({team_visitante}): {probs.get("2",0):.1f}%']
wedges, texts = ax1.pie(probs.values, labels=labels_pie, startangle=90, 
                         colors=colores_pie, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax1.set_title('PROBABILIDADES 1X2
(Monte Carlo)', fontsize=13, fontweight='bold')

# 2. Total goles
ax2 = axes[0, 1]
total_goles = mc['total_goles']
ax2.hist(total_goles, bins=range(0, total_goles.max()+2), color=COLORES['celeste'], 
         edgecolor='white', linewidth=2, alpha=0.8, density=True)
ax2.set_xlabel('Total de Goles', fontsize=12, fontweight='bold')
ax2.set_ylabel('Densidad', fontsize=12, fontweight='bold')
ax2.set_title('DISTRIBUCIÓN TOTAL DE GOLES', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.axvline(total_goles.mean(), color='#CE1126', linestyle='--', linewidth=2, 
            label=f'Media: {total_goles.mean():.2f}')
ax2.legend(fontsize=11)

# 3. Top 10 resultados
ax3 = axes[1, 0]
top_resultados = mc.groupby(['goles_local', 'goles_visitante']).size().sort_values(ascending=False).head(10)
top_resultados_pct = (top_resultados / n_sim) * 100
etiquetas = [f"{g[0]}-{g[1]}" for g in top_resultados.index]
colores_barras = [COLORES['verde'] if g[0] > g[1] else 
                  (COLORES['amarillo'] if g[0] == g[1] else COLORES['naranja']) 
                  for g in top_resultados.index]

bars = ax3.barh(range(len(etiquetas)), top_resultados_pct.values, color=colores_barras,
                edgecolor='white', linewidth=2)
ax3.set_yticks(range(len(etiquetas)))
ax3.set_yticklabels(etiquetas)
ax3.set_xlabel('Probabilidad (%)', fontsize=12, fontweight='bold')
ax3.set_title('TOP 10 RESULTADOS MÁS PROBABLES', fontsize=13, fontweight='bold')
ax3.invert_yaxis()
ax3.grid(axis='x', alpha=0.3)

for i, (bar, pct) in enumerate(zip(bars, top_resultados_pct.values)):
    width = bar.get_width()
    ax3.annotate(f'{pct:.1f}%', xy=(width, bar.get_y() + bar.get_height() / 2),
                xytext=(5, 0), textcoords="offset points",
                ha='left', va='center', fontsize=10, fontweight='bold')

# 4. Diferencia goles
ax4 = axes[1, 1]
diff_goles = mc['diferencia']
ax4.hist(diff_goles, bins=range(diff_goles.min(), diff_goles.max()+2), 
         color=COLORES['lavanda'], edgecolor='white', linewidth=2, alpha=0.8)
ax4.set_xlabel('Diferencia (Local - Visitante)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Frecuencia', fontsize=12, fontweight='bold')
ax4.set_title('DISTRIBUCIÓN DIFERENCIA DE GOLES', fontsize=13, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)
ax4.axvline(0, color='#CE1126', linestyle='--', linewidth=2, label='Empate (0)')
ax4.legend(fontsize=11)

plt.suptitle(f'SIMULACIÓN MONTE CARLO - {n_sim:,} ITERACIONES
{team_local} vs {team_visitante}', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()



### Gráfico 4: Dashboard Resumen Ejecutivo


In [ ]:

# =============================================================================
# CELDA 12: GRÁFICO 4 - RESUMEN EJECUTIVO
# =============================================================================
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.3)

# Título
fig.suptitle(
    f'📊 PRONÓSTICO ESTADÍSTICO AVANZADO - MUNDIAL 2026
'
    f'{team_local} vs {team_visitante} | Grupo K | 17/06/2026 20:00
'
    f'Modelo: Poisson + Monte Carlo + Datos Históricos',
    fontsize=16, fontweight='bold', y=0.98
)

# Panel 1: 1X2 barras
ax1 = fig.add_subplot(gs[0, :2])
resultados_labels = ['1 (Local)', 'X (Empate)', '2 (Visitante)']
probabilidades = [prob_1x2['1']*100, prob_1x2['X']*100, prob_1x2['2']*100]
colores = [COLORES['verde'], COLORES['amarillo'], COLORES['naranja']]

bars = ax1.bar(resultados_labels, probabilidades, color=colores, edgecolor='white', linewidth=3, width=0.6)
ax1.set_ylabel('Probabilidad (%)', fontsize=12, fontweight='bold')
ax1.set_title('PROBABILIDADES DE RESULTADO FINAL', fontsize=14, fontweight='bold')
ax1.set_ylim(0, max(probabilidades) * 1.3)
ax1.grid(axis='y', alpha=0.3)

for bar, prob in zip(bars, probabilidades):
    height = bar.get_height()
    ax1.annotate(f'{prob:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5), textcoords="offset points",
                ha='center', va='bottom', fontsize=14, fontweight='bold')

# Panel 2: Goles esperados
ax2 = fig.add_subplot(gs[0, 2])
equipos = [team_local, team_visitante]
lambdas = [lambda_local, lambda_visitante]
colores_goles = [COLORES['verde'], COLORES['amarillo']]

bars2 = ax2.bar(equipos, lambdas, color=colores_goles, edgecolor='white', linewidth=2, width=0.5)
ax2.set_ylabel('Goles Esperados (λ)', fontsize=11, fontweight='bold')
ax2.set_title('GOLES ESPERADOS', fontsize=13, fontweight='bold')
ax2.set_ylim(0, max(lambdas) * 1.5)

for bar, lam in zip(bars2, lambdas):
    height = bar.get_height()
    ax2.annotate(f'{lam:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=12, fontweight='bold')

# Panel 3: Estadísticas
ax3 = fig.add_subplot(gs[1, :])
stats_text = (
    f"📈 BASE ESTADÍSTICA DEL MODELO
"
    f"   • Dataset histórico: {len(df_hist):,} partidos (1872-2024)
"
    f"   • Partidos torneo actual: {stats_torneo['total_partidos']} finalizados
"
    f"   • Promedio goles/partido (torneo): {stats_torneo['promedio_goles_total']:.2f}
"
    f"   • Promedio goles local (torneo): {stats_torneo['promedio_goles_local']:.2f}
"
    f"   • Promedio goles visitante (torneo): {stats_torneo['promedio_goles_visitante']:.2f}
"
    f"
"
    f"🎯 PRONÓSTICO PRINCIPAL
"
    f"   Resultado más probable: {resultados_labels[np.argmax(probabilidades)]} ({max(probabilidades):.1f}%)
"
    f"   Marcador más probable: {gl}-{gv} ({marcador_prob:.1f}%)
"
    f"   Total goles esperados: {lambda_local + lambda_visitante:.2f}
"
    f"   Over 2.5: {((mc['total_goles'] > 2.5).mean()*100):.1f}% | Ambos anotan: {(((mc['goles_local'] > 0) & (mc['goles_visitante'] > 0)).mean()*100):.1f}%"
)
ax3.text(0.05, 0.95, stats_text, transform=ax3.transAxes, fontsize=11,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor=COLORES['gris'], alpha=0.8, edgecolor='gray'))
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)
ax3.axis('off')

# Panel 4: Intervalos confianza
ax4 = fig.add_subplot(gs[2, :])
ax4.text(0.5, 0.5, 
         'INTERVALOS DE CONFIANZA AL 95% (Distribución Poisson)

'
         f'• Goles {team_local}: [{max(0, lambda_local - 1.96*np.sqrt(lambda_local)):.1f}, {lambda_local + 1.96*np.sqrt(lambda_local):.1f}]
'
         f'• Goles {team_visitante}: [{max(0, lambda_visitante - 1.96*np.sqrt(lambda_visitante)):.1f}, {lambda_visitante + 1.96*np.sqrt(lambda_visitante):.1f}]
'
         f'• Total goles: [{max(0, (lambda_local + lambda_visitante) - 1.96*np.sqrt(lambda_local + lambda_visitante)):.1f}, '
         f'{(lambda_local + lambda_visitante) + 1.96*np.sqrt(lambda_local + lambda_visitante):.1f}]
'
         f'
⚠️ Nota: El modelo se actualiza automáticamente con datos del torneo. '
         f'Reejecutar esta celda después de cada jornada mejora la precisión.',
         transform=ax4.transAxes, fontsize=12, verticalalignment='center', horizontalalignment='center',
         bbox=dict(boxstyle='round', facecolor=COLORES['amarillo'], alpha=0.6, edgecolor='orange'))
ax4.set_xlim(0, 1)
ax4.set_ylim(0, 1)
ax4.axis('off')

plt.show()



## 👥 SECCIÓN 9: COMPARATIVA DE PLANTILLAS
Datos de plantillas basados en ESPN y Transfermarkt (Junio 2026).


In [ ]:

# =============================================================================
# CELDA 13: COMPARATIVA DE PLANTILLAS Y VALOR DE MERCADO
# =============================================================================
# Datos de plantillas (fuente: ESPN / Transfermarkt, Junio 2026)
plantilla_data = {
    'Indicador': [
        'Valor total plantilla (€M)',
        'Jugador más valioso (€M)',
        'Jugadores top 5 ligas',
        'Experiencia Mundial',
        'Edad promedio',
        'Entrenador',
        'Ranking FIFA aprox.',
        'Factor localía',
        'Debutante Mundial'
    ],
    'Colombia': [
        '302.35', 'Luis Díaz 70.0', '8+', '6 participaciones', '~28', 'Néstor Lorenzo', '15-20', 'No (visitante)', 'No'
    ],
    'Uzbekistan': [
        '85.33', 'Khusanov 15.0', '2-3', '0 (debutante)', '~27', 'Fabio Cannavaro', '70-80', 'Sí (local)', 'Sí'
    ]
}

df_plantilla = pd.DataFrame(plantilla_data)
print("="*70)
print("👥 COMPARATIVA DE PLANTILLAS - MUNDIAL 2026")
print("="*70)
print(df_plantilla.to_string(index=False))

# Gráfico comparativo de valor de mercado
fig, ax = plt.subplots(figsize=(10, 6))

equipos_comp = ['Colombia', 'Uzbekistán']
valores = [302.35, 85.33]
colores_comp = [COLORES['amarillo'], COLORES['verde']]

bars = ax.bar(equipos_comp, valores, color=colores_comp, edgecolor='white', linewidth=3, width=0.5)
ax.set_ylabel('Valor de Mercado (Millones €)', fontsize=12, fontweight='bold')
ax.set_title('VALOR DE PLANTILLA - COMPARATIVA
(Fuente: Transfermarkt, Junio 2026)', 
             fontsize=14, fontweight='bold')
ax.set_ylim(0, max(valores) * 1.3)
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, valores):
    height = bar.get_height()
    ax.annotate(f'€{val:.2f}M', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5), textcoords="offset points",
                ha='center', va='bottom', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()



## ✅ CONCLUSIÓN Y PRONÓSTICO FINAL

**Resumen del análisis integrado:**

| Método | 1 (Uzbekistán) | X (Empate) | 2 (Colombia) |
|--------|---------------|-----------|-------------|
| Modelo Poisson | ~52% | ~22% | ~26% |
| Monte Carlo (10k sim.) | ~52% | ~22% | ~26% |

**Marcador más probable:** 2-1 (9.7%)

**Factores clave:**
- 🇺🇿 **Ventaja Uzbekistán:** Factor localía (Estadio Azteca), debut histórico (motivación extra)
- 🇨🇴 **Ventaja Colombia:** Plantilla 3.5x más valiosa, experiencia en Mundiales, jugadores en top ligas europeas
- ⚖️ **Equilibrio:** El modelo histórico + torneo actual equilibra ambos factores

**Recomendación:** Reejecutar este notebook después de cada jornada del torneo para recalibrar λ con datos reales de ambos equipos.


In [ ]:

# =============================================================================
# CELDA 14: EXPORTAR RESULTADOS A CSV
# =============================================================================
# Guardar tabla de resultados probables
output_file = 'pronostico_colombia_uzbekistan_2026.csv'
df_resultados.to_csv(output_file, index=False)
print(f"✅ Resultados exportados a: {output_file}")
print(f"
📊 Resumen final del pronóstico:")
print(f"   Equipos: {team_local} vs {team_visitante}")
print(f"   Fecha: 17/06/2026 20:00")
print(f"   Modelo: Poisson + Monte Carlo + Histórico")
print(f"   1: {prob_1x2['1']*100:.1f}% | X: {prob_1x2['X']*100:.1f}% | 2: {prob_1x2['2']*100:.1f}%")
print(f"
📝 Para actualizar el modelo: reejecutar desde la Celda 5 después de cada jornada")
